## MountainCar - QTable, state aggregation


Q-Learning example using OpenAI gym MountainCar environment

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import wrappers

n_states = 50         # for state aggregation
num_episodes = 10_000 # number of episodes for training
num_eval     = 10_000 # number of episodes for evaluation (scoring)

initial_lr = 1.0 # Learning rate
min_lr     = 0.003
gamma      = 1.0
t_max      = 10_000 # maximum steps allowed per episode
eps        = 0.02

def run_episode(env, policy=None, render=False):
    obs, _ = env.reset()
    total_reward = 0
    for t in range(t_max):
        if render: env.render()
        if policy is None:
            action = env.action_space.sample()
        else:
            a,b = obs_to_state(env, obs)
            action = policy[a][b]
        obs, reward, terminated, truncated, _ = env.step(action)
        # done = terminated or truncated
        done = terminated
        total_reward += gamma ** t * reward
        if done: break
    return total_reward

def obs_to_state(env, obs):
    """ Maps an observation to agent's state """
    env_low  = env.observation_space.low
    env_high = env.observation_space.high
    env_dx   = (env_high - env_low) / n_states
    x = int((obs[0] - env_low[0])/env_dx[0])
    v = int((obs[1] - env_low[1])/env_dx[1])
    return x, v

def state_to_obs(env, state):
    """ Maps back agent's state to observation"""
    env_low  = env.observation_space.low
    env_high = env.observation_space.high
    env_dx   = (env_high - env_low) / n_states
    X = env_low[0] + state[0]*env_dx[0]
    V = env_low[1] + state[1]*env_dx[1]
    return X, V
    
if __name__ == '__main__':
    env_name = 'MountainCar-v0'
    env = gym.make(env_name)
    gym.utils.seeding.np_random(seed=0) # env.seed(0)
    np.random.seed(0)
    print (f'----- using Q-Learning with state aggregation {n_states} -----')
    # q_table = np.zeros((n_states, n_states, 3))
    q_table = np.random.rand(n_states, n_states, 3)*1.0
    episode_rewards = []
    for i in range(num_episodes):
        obs, _ = env.reset()
        total_reward = 0
        ## alpha: learning rate is decreased at each step
        alpha = max(min_lr, initial_lr * (0.85 ** (i//100)))
        for j in range(t_max):
            x, v = obs_to_state(env, obs)
            
            # policy for training:
            # not epsilon greedy...
            # epsilon + softmax sampling
            if np.random.uniform(0, 1) < eps:
                action = np.random.choice(env.action_space.n)
            else:
                logits = q_table[x][v]
                # softmax...
                logits_exp = np.exp(logits)
                probs = logits_exp / np.sum(logits_exp)
                action = np.random.choice(env.action_space.n, p=probs)
                
            obs, reward, terminated, truncated, _ = env.step(action)
            # done = terminated or truncated
            done = terminated
            total_reward += reward
            # update q table
            x_, v_ = obs_to_state(env, obs)
            q_table[x][v][action] = q_table[x][v][action] + alpha * (reward + gamma *  np.max(q_table[x_][v_]) - q_table[x][v][action])
            if done:
                episode_rewards.append(total_reward)
                break
        if i % 100 == 0:
            print('\rIteration #%d -- episode reward = %d.' %(i+1, total_reward),' '*20,end="")
    #-------------------------------------------        
    solution_policy = np.argmax(q_table, axis=2) # final policy for inference (greedy)
    #-------------------------------------------        
    
    # Evaluate it
    solution_policy_scores = [run_episode(env, solution_policy, False) for _ in range(num_eval)]
    print("Average score of solution = ", np.mean(solution_policy_scores))

    env.close()
    
    # Animate it
    env = gym.make(env_name, render_mode="human")
    run_episode(env, solution_policy, True)
    env.close()

----- using Q-Learning with state aggregation 50 -----
Iteration #9901 -- episode reward = -168.                     Average score of solution =  -117.2274


: 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Example list of values
values = solution_policy_scores

# Validate input
if not isinstance(values, (list, np.ndarray)):
    raise TypeError("Values must be a list or NumPy array.")
if len(values) == 0:
    raise ValueError("Values list cannot be empty.")
if not all(isinstance(v, (int, float)) for v in values):
    raise ValueError("All elements in values must be numeric.")

# Create the histogram with KDE
plt.figure(figsize=(8, 5))
sns.histplot(values, kde=True, bins='auto', color='skyblue', edgecolor='black')

# Add labels and title
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Histogram with KDE")

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_smooth_with_std(values, window=50):
    """
    Plots a moving average with ±1 standard deviation shaded area.
    
    Parameters:
        values (list or array): Input numeric values.
        window (int): Window size for moving average and std.
    """
    # Validate input
    if not isinstance(values, (list, np.ndarray, pd.Series)):
        raise TypeError("Values must be a list, numpy array, or pandas Series.")
    if len(values) < window:
        raise ValueError("Length of values must be at least equal to the window size.")
    if window <= 0:
        raise ValueError("Window size must be positive.")

    # Convert to pandas Series for rolling calculations
    series = pd.Series(values)

    # Compute rolling mean and standard deviation
    rolling_mean = series.rolling(window=window, center=True).mean()
    rolling_std = series.rolling(window=window, center=True).std()

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(series.index, rolling_mean, color='blue', label='Moving Average')
    plt.fill_between(
        series.index,
        rolling_mean - rolling_std,
        rolling_mean + rolling_std,
        color='blue',
        alpha=0.2,
        label='±1 Std Dev'
    )
    plt.plot(series.index, series, color='gray', alpha=0.4, label='Original Data')
    plt.ylim(-500,-100)

    plt.title(f"Moving Average (window={window}) with ±1 Std Dev")
    plt.xlabel("Index")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Example usage:
if __name__ == "__main__":
    """
    # Generate example noisy data
    np.random.seed(42)
    x = np.linspace(0, 10, 500)
    y = np.sin(x) + np.random.normal(0, 0.3, size=len(x))
    """
    y = episode_rewards
    plot_smooth_with_std(y, window=100)

In [ ]:
from scipy.special import softmax
env = gym.make(env_name)
fig, ax = plt.subplots(figsize=(6,6))

data = softmax(q_table,axis=2)
data = solution_policy
data = np.max(q_table,axis=2)

# Show image with a colormap
im = ax.imshow(data, cmap='coolwarm', origin='lower') # cmap='viridis' 'coolwarm'

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.7)
cbar.set_label("Intensity", rotation=270, labelpad=15)

# xticks = ax.get_xticks()
xticks = np.array(range(n_states//10+1))*10; xticks[-1]-=1
print(xticks)
xticks_labels = [f'{state_to_obs(env,(tx,0))[0]:.2f}' for tx in xticks]
ax.set_xticks(xticks)
ax.set_xticklabels(xticks_labels)

yticks = xticks
yticks_labels = [f'{state_to_obs(env,(0,ty))[1]:.2f}' for ty in yticks]
ax.set_yticks(yticks)
ax.set_yticklabels(yticks_labels)

x,v = obs_to_state(env,(-0.5,0))
ax.set_yticklabels(yticks_labels)
plt.axvline(x=x,color='white',linestyle='--',linewidth=2, label=f'x={x}')
plt.axhline(y=v,color='white',linestyle='--',linewidth=2, label=f'y={v}')

# Set title and axis labels
ax.set_title('Values')
ax.set_xlabel("X Position")
ax.set_ylabel("Y Speed")

plt.tight_layout()
plt.show()

plt.show()
